<a href="https://colab.research.google.com/github/NabilaAzzahra/rancangan-arsitektur-solusi-AI/blob/main/poc_rag_bank_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PoC RAG Layanan Informasi Nasabah (PT Bank Nusantara Mandiri, fiktif)

Notebook ini menjalankan eksperimen Bab 5: **B0** (LLM tanpa RAG) melawan **R1** (RAG: bge-m3, ChromaDB, chunk 500 token, overlap 10%, K = 5, klasifikasi intent, ambang skor retrieval) pada 30 pertanyaan uji.

**Langkah singkat**
1. Buat API key di Google AI Studio, lalu simpan di Colab: ikon kunci (Secrets) di kiri, nama `GOOGLE_API_KEY`, aktifkan *Notebook access*.
2. Unggah `poc_data.zip` ke panel File Colab (atau jalankan sel unggah di bawah).
3. Jalankan **Runtime > Run all**. Sel terakhir mengunduh `hasil_poc.zip`.
4. Isi kolom `nilai_B0` dan `nilai_R1` di `lembar_penilaian.csv` dengan **benar**, **sebagian**, atau **salah**, lalu kirim `hasil_poc.zip` dan lembar yang sudah terisi.

Seluruh dokumen dan pertanyaan bersifat **fiktif**. Parameter pada sel konfigurasi ditetapkan sebelum pengukuran dan tidak diubah setelah melihat hasil.

In [ ]:
!pip -q install google-genai sentence-transformers chromadb "ragas>=0.2.10,<0.4" langchain-google-genai langchain-huggingface datasets mlflow pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os, re, time, json, glob, zipfile, getpass, random
import numpy as np
import pandas as pd

# ---- Konfigurasi (ditetapkan sebelum pengukuran; jangan diubah setelah melihat hasil) ----
MODEL = "gemini-3.5-flash-lite"  # model pengganti gemini-2.5-flash (tidak lagi tersedia untuk akun baru)
JUDGE_MODEL = MODEL             # LLM penilai untuk Ragas (keterbatasan: sama dengan generator)
TEMPERATURE = 0.2
THINKING = None                 # memakai bawaan model (3.5 Flash-Lite: thinking minimal); jangan diubah
K = 5
CHUNK_TOKENS = 500
OVERLAP = 0.10
SCORE_THRESHOLD = 0.30          # ambang skor kosinus retrieval (parameter penulis); konservatif karena topik di luar cakupan sudah disaring klasifikasi intent
REQUEST_SLEEP = 0.0             # naikkan (mis. 6) bila muncul galat 429 / kuota
DATA_DIR = "poc"
random.seed(0); np.random.seed(0)

try:
    from google.colab import userdata
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    API_KEY = os.environ.get("GOOGLE_API_KEY") or getpass.getpass("GOOGLE_API_KEY: ")
os.environ["GOOGLE_API_KEY"] = API_KEY
print("Konfigurasi siap:", MODEL, "K =", K, "ambang =", SCORE_THRESHOLD)
assert "2.5" not in MODEL, "Model 2.5 tidak lagi tersedia untuk akun baru; pakai gemini-3.5-flash-lite atau gemini-3.8-flash"

Konfigurasi siap: gemini-3.5-flash-lite K = 5 ambang = 0.3


In [ ]:
# Unggah dan ekstrak data (lewati bila folder 'poc' sudah ada)
if not os.path.isdir(DATA_DIR):
    if not os.path.exists("poc_data.zip"):
        from google.colab import files
        files.upload()
    with zipfile.ZipFile("poc_data.zip") as z:
        z.extractall(".")
print(sorted(os.listdir(DATA_DIR)))
print(sorted(os.listdir(os.path.join(DATA_DIR, "docs"))))

['docs', 'pertanyaan_uji.csv']
['DOC-001_tarif_tabungan_kartu_debit.md', 'DOC-002_syarat_kredit.md', 'DOC-003_prosedur_kartu.md', 'DOC-004_faq.md']


## 1. Dokumen, chunking, dan indeks

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("BAAI/bge-m3")

def tok_len(text):
    try:
        return len(embedder.tokenizer.encode(text, add_special_tokens=False))
    except Exception:
        return int(len(text.split()) * 1.3)

def embed(texts):
    return embedder.encode(list(texts), normalize_embeddings=True, show_progress_bar=False).tolist()

def parse_doc(path):
    txt = open(path, encoding="utf-8").read()
    head, _, body = txt.partition("\n## ")
    meta = {}
    for line in head.splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            meta[k.strip()] = v.strip()
    sections = []
    for part in re.split(r"\n(?=## )", "## " + body):
        part = part.strip()
        if not part:
            continue
        title, _, text = part.partition("\n")
        sections.append((title[3:].strip(), text.strip()))
    return meta, sections

def split_long(text, max_tokens=CHUNK_TOKENS, overlap=OVERLAP):
    if tok_len(text) <= max_tokens:
        return [text]
    sents = re.split(r"(?<=[.!?])\s+", text)
    out, cur = [], []
    for s in sents:
        cur.append(s)
        if tok_len(" ".join(cur)) > max_tokens and len(cur) > 1:
            cur.pop()
            out.append(" ".join(cur))
            keep, n = [], 0
            for t in reversed(cur):
                n += tok_len(t)
                if n > max_tokens * overlap:
                    break
                keep.insert(0, t)
            cur = keep + [s]
    if cur:
        out.append(" ".join(cur))
    return out

CHUNKS = []
for path in sorted(glob.glob(os.path.join(DATA_DIR, "docs", "*.md"))):
    meta, secs = parse_doc(path)
    for title, text in secs:
        label = title.split(".")[0].strip()          # mis. "Pasal 2" atau "FAQ 1"
        for j, piece in enumerate(split_long(text)):
            cid = f"{meta['doc_id']}#{label}" + (f"-{j+1}" if j else "")
            header = f"[{meta['doc_id']} v{meta['versi']} | {meta['jenis']} | {label}] {title}"
            CHUNKS.append({"chunk_id": cid, "doc_id": meta["doc_id"], "versi": meta["versi"],
                           "jenis": meta["jenis"], "label": label, "text": header + "\n" + piece})
CHUNK_TEXT = {c["chunk_id"]: c["text"] for c in CHUNKS}
print(len(CHUNKS), "chunk;", "maks token per chunk =", max(tok_len(c["text"]) for c in CHUNKS))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

23 chunk; maks token per chunk = 89


In [ ]:
import chromadb
chroma = chromadb.Client()
col = chroma.create_collection("poc", metadata={"hnsw:space": "cosine"})
col.add(ids=[c["chunk_id"] for c in CHUNKS],
        documents=[c["text"] for c in CHUNKS],
        embeddings=embed([c["text"] for c in CHUNKS]),
        metadatas=[{"doc_id": c["doc_id"], "jenis": c["jenis"], "label": c["label"]} for c in CHUNKS])

def retrieve(q, k=K):
    r = col.query(query_embeddings=embed([q]), n_results=k)
    ids = r["ids"][0]; docs = r["documents"][0]
    scores = [1.0 - d for d in r["distances"][0]]      # jarak kosinus -> kemiripan
    return docs, ids, scores
print("Indeks siap:", col.count(), "chunk")

Indeks siap: 23 chunk


## 2. LLM, prompt, dan dua konfigurasi

In [ ]:
from google import genai
from google.genai import types
client = genai.Client(api_key=API_KEY)

USE_TEMP = True   # akan dimatikan otomatis bila model menolak parameter temperature

def gemini(prompt, system=None, temperature=TEMPERATURE, retries=6):
    global USE_TEMP
    for a in range(retries):
        kw = {"system_instruction": system}
        if USE_TEMP: kw["temperature"] = temperature
        cfg = types.GenerateContentConfig(**kw)
        try:
            if REQUEST_SLEEP: time.sleep(REQUEST_SLEEP)
            r = client.models.generate_content(model=MODEL, contents=prompt, config=cfg)
            u = r.usage_metadata
            tin = int(getattr(u, "prompt_token_count", 0) or 0)
            tout = int((getattr(u, "candidates_token_count", 0) or 0) + (getattr(u, "thoughts_token_count", 0) or 0))
            return (r.text or "").strip(), tin, tout
        except Exception as e:
            msg = str(e)
            if "404" in msg or "NOT_FOUND" in msg:
                print("MODEL TIDAK DITEMUKAN:", MODEL, "-", msg[:200])
                try:
                    print("Model yang tersedia untuk key ini:")
                    for m in client.models.list():
                        acts = getattr(m, "supported_actions", None) or []
                        if "generateContent" in acts: print("  ", m.name)
                except Exception as e2:
                    print("(gagal menampilkan daftar model)", str(e2)[:100])
                raise RuntimeError(f"Ubah MODEL di sel konfigurasi ke salah satu model di atas. ({MODEL} tidak tersedia)")
            if USE_TEMP and "temperature" in msg.lower():
                USE_TEMP = False; print("Model menolak temperature; dijalankan tanpa temperature (catat di 5.5)."); continue
            print("coba ulang", a + 1, "->", msg[:140]); time.sleep(3 * 2 ** a)
    raise RuntimeError("Panggilan Gemini gagal berulang")

INTENT_SYS = '''Klasifikasikan pesan nasabah PT Bank Nusantara Mandiri ke tepat satu label:
- produk: pertanyaan informasi produk, tarif, syarat, prosedur, atau jam layanan (tabungan, kartu debit, KTA, KPR, kartu kredit).
- sensitif: keluhan, sengketa transaksi, dugaan penipuan, permintaan saldo atau data rekening, atau permintaan data pribadi.
- di_luar: selain itu (topik umum, investasi, bank lain, hiburan).
Jawab hanya dengan satu kata: produk, sensitif, atau di_luar.'''

GEN_SYS = '''Kamu asisten virtual PT Bank Nusantara Mandiri (fiktif). Jawab pertanyaan nasabah dalam bahasa Indonesia yang ringkas dan sopan.
Aturan:
1. Jawab HANYA berdasarkan konteks yang diberikan. Jangan memakai pengetahuan lain.
2. Bila konteks memuat dokumen resmi dan FAQ yang bertentangan, gunakan dokumen resmi.
3. Cantumkan sumber di akhir jawaban dengan format [DOC-xxx Pasal n] atau [DOC-xxx FAQ n].
4. Jangan memberi rekomendasi keuangan atau keputusan kredit.
5. Bila konteks tidak cukup untuk menjawab, balas persis: TIDAK_TAHU'''

B0_SYS = '''Kamu asisten virtual PT Bank Nusantara Mandiri. Jawab pertanyaan nasabah dalam bahasa Indonesia yang ringkas dan sopan.'''

MSG_ESKALASI = "Pertanyaan ini perlu ditangani petugas kami. Percakapan Anda akan dialihkan ke petugas."
MSG_TOLAK = "Maaf, saya hanya dapat membantu informasi produk dan layanan PT Bank Nusantara Mandiri. Silakan hubungi kanal resmi bank."
MSG_TIDAK_TAHU = "Maaf, saya belum menemukan informasi yang cukup untuk menjawab. Percakapan Anda akan dialihkan ke petugas."

def classify(q):
    out, i, o = gemini(q, system=INTENT_SYS, temperature=0.0)
    low = out.lower()
    for lab in ("sensitif", "di_luar", "produk"):
        if lab in low:
            return lab, i, o
    return "produk", i, o

def build_prompt(q, ctx):
    return "Konteks:\n" + "\n\n".join(ctx) + f"\n\nPertanyaan nasabah: {q}\nJawaban:"

def run_b0(q):
    t0 = time.perf_counter()
    ans, i, o = gemini(q, system=B0_SYS)
    return {"jawaban": ans, "aksi": "jawab", "intent": "-", "konteks_ids": "", "skor_top1": np.nan,
            "latensi_s": time.perf_counter() - t0, "token_in": i, "token_out": o}

def run_r1(q):
    t0 = time.perf_counter(); tin = tout = 0
    lab, i, o = classify(q); tin += i; tout += o
    ids, scores, ctx = [], [], []
    if lab == "sensitif":
        aksi, ans = "eskalasi", MSG_ESKALASI
    elif lab == "di_luar":
        aksi, ans = "tolak", MSG_TOLAK
    else:
        ctx, ids, scores = retrieve(q)
        if not ctx or scores[0] < SCORE_THRESHOLD:
            aksi, ans = "eskalasi", MSG_TIDAK_TAHU
        else:
            ans, i, o = gemini(build_prompt(q, ctx), system=GEN_SYS); tin += i; tout += o
            if "TIDAK_TAHU" in ans:
                aksi, ans = "eskalasi", MSG_TIDAK_TAHU
            else:
                aksi = "jawab"
    return {"jawaban": ans, "aksi": aksi, "intent": lab, "konteks_ids": "|".join(ids),
            "skor_top1": scores[0] if scores else np.nan, "konteks_teks": ctx,
            "latensi_s": time.perf_counter() - t0, "token_in": tin, "token_out": tout}
print("Fungsi siap")

Fungsi siap


## 3. Menjalankan 30 pertanyaan uji

In [ ]:
Q = pd.read_csv(os.path.join(DATA_DIR, "pertanyaan_uji.csv"), keep_default_na=False)
assert len(Q) == 30, len(Q)
rows = []
for n, r in Q.iterrows():
    b0 = run_b0(r["pertanyaan"]); r1 = run_r1(r["pertanyaan"])
    rows.append({"id": r["id"], "tipe": r["tipe"], "aksi_diharapkan": r["aksi_diharapkan"], "B0": b0, "R1": r1})
    print(f'{r["id"]} {r["tipe"][:8]:8} R1={r1["aksi"]:8} {r1["latensi_s"]:.1f}s  skor={r1["skor_top1"]}')

flat = []
for x in rows:
    for cfg in ("B0", "R1"):
        d = {k: v for k, v in x[cfg].items() if k != "konteks_teks"}
        d.update({"id": x["id"], "tipe": x["tipe"], "aksi_diharapkan": x["aksi_diharapkan"], "konfigurasi": cfg})
        flat.append(d)
RAW = pd.DataFrame(flat)
RAW["token_total"] = RAW["token_in"] + RAW["token_out"]
RAW.to_csv("hasil_mentah.csv", index=False, encoding="utf-8-sig")
print("Disimpan: hasil_mentah.csv")

Q01 dalam_ca R1=jawab    1.8s  skor=0.7846360206604004
Q02 dalam_ca R1=jawab    1.8s  skor=0.7883076667785645
Q03 dalam_ca R1=jawab    1.4s  skor=0.8219338655471802
Q04 dalam_ca R1=jawab    1.3s  skor=0.6411563754081726
Q05 dalam_ca R1=jawab    1.6s  skor=0.7777506709098816
coba ulang 1 -> 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For
coba ulang 2 -> 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For
coba ulang 3 -> 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For
Q06 dalam_ca R1=jawab    23.1s  skor=0.7534356117248535
coba ulang 1 -> 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For
coba ulang 2 -> 429 RESOURCE_EXHAUSTED. {

In [ ]:
# ---- Perbaikan kompatibilitas Ragas dan langchain-community (jalankan SEBELUM sel Ragas) ----
import sys, types, importlib

def _stub_vertex():
    try:
        import langchain_community
    except Exception:
        return "langchain_community tidak terpasang"
    notes = []
    try:
        importlib.import_module("langchain_community.chat_models.vertexai")
    except Exception:
        m = types.ModuleType("langchain_community.chat_models.vertexai")
        m.ChatVertexAI = type("ChatVertexAI", (), {})
        sys.modules["langchain_community.chat_models.vertexai"] = m
        notes.append("stub chat_models.vertexai")
    try:
        import langchain_community.llms as _L
        if not hasattr(_L, "VertexAI"):
            _L.VertexAI = type("VertexAI", (), {})
            notes.append("stub llms.VertexAI")
    except Exception:
        pass
    return ", ".join(notes) or "tidak perlu stub"

print("Perbaikan:", _stub_vertex())
for k in [k for k in list(sys.modules) if k == "ragas" or k.startswith("ragas.")]:
    del sys.modules[k]                      # buang impor ragas yang setengah jadi

# Bila data hasil (variabel 'rows') hilang karena sesi dimulai ulang, bangun dari hasil_mentah.csv
if "rows" not in globals():
    _raw = pd.read_csv("hasil_mentah.csv", keep_default_na=False)
    Q = pd.read_csv(os.path.join(DATA_DIR, "pertanyaan_uji.csv"), keep_default_na=False)
    rows = []
    for _, q in Q.iterrows():
        x = {"id": q["id"], "tipe": q["tipe"], "aksi_diharapkan": q["aksi_diharapkan"]}
        for cfg in ("B0", "R1"):
            d = _raw[(_raw["id"] == q["id"]) & (_raw["konfigurasi"] == cfg)].iloc[0]
            x[cfg] = {"jawaban": d["jawaban"], "aksi": d["aksi"],
                      "konteks_teks": [CHUNK_TEXT[c] for c in str(d["konteks_ids"]).split("|") if c in CHUNK_TEXT]}
        rows.append(x)
    RAW = pd.read_csv("hasil_mentah.csv")
    RAW["token_total"] = RAW["token_in"] + RAW["token_out"]
    print("rows dibangun ulang dari hasil_mentah.csv:", len(rows))
else:
    print("rows sudah ada di memori:", len(rows))


Perbaikan: stub chat_models.vertexai
rows sudah ada di memori: 30


## 4. Metrik kualitas dengan Ragas

In [ ]:
# ===== SEL PEMULIHAN: jalankan sekali setelah runtime di-restart, SEBELUM sel Ragas =====
import os, re, time, json, glob, zipfile, getpass, random
import numpy as np
import pandas as pd

# ---- Konfigurasi (ditetapkan sebelum pengukuran; jangan diubah setelah melihat hasil) ----
MODEL = "gemini-3.5-flash-lite"  # model pengganti gemini-2.5-flash (tidak lagi tersedia untuk akun baru)
JUDGE_MODEL = MODEL             # LLM penilai untuk Ragas (keterbatasan: sama dengan generator)
TEMPERATURE = 0.2
THINKING = None                 # memakai bawaan model (3.5 Flash-Lite: thinking minimal); jangan diubah
K = 5
CHUNK_TOKENS = 500
OVERLAP = 0.10
SCORE_THRESHOLD = 0.30          # ambang skor kosinus retrieval (parameter penulis); konservatif karena topik di luar cakupan sudah disaring klasifikasi intent
REQUEST_SLEEP = 0.0             # naikkan (mis. 6) bila muncul galat 429 / kuota
DATA_DIR = "poc"
random.seed(0); np.random.seed(0)

try:
    from google.colab import userdata
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    API_KEY = os.environ.get("GOOGLE_API_KEY") or getpass.getpass("GOOGLE_API_KEY: ")
os.environ["GOOGLE_API_KEY"] = API_KEY
print("Konfigurasi siap:", MODEL, "K =", K, "ambang =", SCORE_THRESHOLD)
assert "2.5" not in MODEL, "Model 2.5 tidak lagi tersedia untuk akun baru; pakai gemini-3.5-flash-lite atau gemini-3.8-flash"
# Unggah dan ekstrak data (lewati bila folder 'poc' sudah ada)
if not os.path.isdir(DATA_DIR):
    if not os.path.exists("poc_data.zip"):
        from google.colab import files
        files.upload()
    with zipfile.ZipFile("poc_data.zip") as z:
        z.extractall(".")
print(sorted(os.listdir(DATA_DIR)))
print(sorted(os.listdir(os.path.join(DATA_DIR, "docs"))))
from transformers import AutoTokenizer
try:
    _tok = AutoTokenizer.from_pretrained("BAAI/bge-m3")
except Exception:
    _tok = None
class _E: pass
embedder = _E(); embedder.tokenizer = _tok

def tok_len(text):
    try:
        return len(embedder.tokenizer.encode(text, add_special_tokens=False))
    except Exception:
        return int(len(text.split()) * 1.3)

def parse_doc(path):
    txt = open(path, encoding="utf-8").read()
    head, _, body = txt.partition("\n## ")
    meta = {}
    for line in head.splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            meta[k.strip()] = v.strip()
    sections = []
    for part in re.split(r"\n(?=## )", "## " + body):
        part = part.strip()
        if not part:
            continue
        title, _, text = part.partition("\n")
        sections.append((title[3:].strip(), text.strip()))
    return meta, sections

def split_long(text, max_tokens=CHUNK_TOKENS, overlap=OVERLAP):
    if tok_len(text) <= max_tokens:
        return [text]
    sents = re.split(r"(?<=[.!?])\s+", text)
    out, cur = [], []
    for s in sents:
        cur.append(s)
        if tok_len(" ".join(cur)) > max_tokens and len(cur) > 1:
            cur.pop()
            out.append(" ".join(cur))
            keep, n = [], 0
            for t in reversed(cur):
                n += tok_len(t)
                if n > max_tokens * overlap:
                    break
                keep.insert(0, t)
            cur = keep + [s]
    if cur:
        out.append(" ".join(cur))
    return out

CHUNKS = []
for path in sorted(glob.glob(os.path.join(DATA_DIR, "docs", "*.md"))):
    meta, secs = parse_doc(path)
    for title, text in secs:
        label = title.split(".")[0].strip()          # mis. "Pasal 2" atau "FAQ 1"
        for j, piece in enumerate(split_long(text)):
            cid = f"{meta['doc_id']}#{label}" + (f"-{j+1}" if j else "")
            header = f"[{meta['doc_id']} v{meta['versi']} | {meta['jenis']} | {label}] {title}"
            CHUNKS.append({"chunk_id": cid, "doc_id": meta["doc_id"], "versi": meta["versi"],
                           "jenis": meta["jenis"], "label": label, "text": header + "\n" + piece})
CHUNK_TEXT = {c["chunk_id"]: c["text"] for c in CHUNKS}
print(len(CHUNKS), "chunk;", "maks token per chunk =", max(tok_len(c["text"]) for c in CHUNKS))
Q = pd.read_csv(os.path.join(DATA_DIR, "pertanyaan_uji.csv"), keep_default_na=False)
# ---- Perbaikan kompatibilitas Ragas dan langchain-community (jalankan SEBELUM sel Ragas) ----
import sys, types, importlib

def _stub_vertex():
    try:
        import langchain_community
    except Exception:
        return "langchain_community tidak terpasang"
    notes = []
    try:
        importlib.import_module("langchain_community.chat_models.vertexai")
    except Exception:
        m = types.ModuleType("langchain_community.chat_models.vertexai")
        m.ChatVertexAI = type("ChatVertexAI", (), {})
        sys.modules["langchain_community.chat_models.vertexai"] = m
        notes.append("stub chat_models.vertexai")
    try:
        import langchain_community.llms as _L
        if not hasattr(_L, "VertexAI"):
            _L.VertexAI = type("VertexAI", (), {})
            notes.append("stub llms.VertexAI")
    except Exception:
        pass
    return ", ".join(notes) or "tidak perlu stub"

print("Perbaikan:", _stub_vertex())
for k in [k for k in list(sys.modules) if k == "ragas" or k.startswith("ragas.")]:
    del sys.modules[k]                      # buang impor ragas yang setengah jadi

# Bila data hasil (variabel 'rows') hilang karena sesi dimulai ulang, bangun dari hasil_mentah.csv
if "rows" not in globals():
    _raw = pd.read_csv("hasil_mentah.csv", keep_default_na=False)
    rows = []
    for _, q in Q.iterrows():
        x = {"id": q["id"], "tipe": q["tipe"], "aksi_diharapkan": q["aksi_diharapkan"]}
        for cfg in ("B0", "R1"):
            d = _raw[(_raw["id"] == q["id"]) & (_raw["konfigurasi"] == cfg)].iloc[0]
            x[cfg] = {"jawaban": d["jawaban"], "aksi": d["aksi"],
                      "konteks_teks": [CHUNK_TEXT[c] for c in str(d["konteks_ids"]).split("|") if c in CHUNK_TEXT]}
        rows.append(x)
    RAW = pd.read_csv("hasil_mentah.csv")
    RAW["token_total"] = RAW["token_in"] + RAW["token_out"]
    print("rows dibangun ulang dari hasil_mentah.csv:", len(rows))
else:
    print("rows sudah ada di memori:", len(rows))

Q = pd.read_csv(os.path.join(DATA_DIR, "pertanyaan_uji.csv"), keep_default_na=False)
print('Q:', len(Q), '| chunk:', len(CHUNK_TEXT), '| rows:', len(rows))


Konfigurasi siap: gemini-3.5-flash-lite K = 5 ambang = 0.3
['docs', 'pertanyaan_uji.csv']
['DOC-001_tarif_tabungan_kartu_debit.md', 'DOC-002_syarat_kredit.md', 'DOC-003_prosedur_kartu.md', 'DOC-004_faq.md']
23 chunk; maks token per chunk = 89
Perbaikan: tidak perlu stub
rows dibangun ulang dari hasil_mentah.csv: 30
Q: 30 | chunk: 23 | rows: 30


In [ ]:
   !pip -q install "ragas>=0.2.10,<0.4" "langchain-community>=0.4,<0.4.2" langchain-google-genai langchain-huggingface sentence-transformers datasets mlflow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.0.0 which is incompatible.


In [ ]:
import warnings, logging
warnings.filterwarnings("ignore")                       # sembunyikan ribuan peringatan 'temperature will be ignored'
logging.getLogger("ragas.executor").setLevel(logging.CRITICAL)

def gold_texts(cids):
    return [CHUNK_TEXT[c] for c in str(cids).split("|") if c in CHUNK_TEXT]

QI = Q.set_index("id")
from ragas import evaluate, EvaluationDataset, SingleTurnSample, RunConfig
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference, LLMContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings

judge = LangchainLLMWrapper(ChatGoogleGenerativeAI(model=JUDGE_MODEL))
lc_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="BAAI/bge-m3", encode_kwargs={"normalize_embeddings": True}))
# timeout longgar + retry terbatas: kuota gratis sering membuat panggilan menunggu lama
rc = RunConfig(max_workers=3, timeout=900, max_retries=5, max_wait=30)

def make_samples(cfg):
    ids, smp = [], []
    for x in rows:
        if x["tipe"] != "dalam_cakupan": continue
        res = x[cfg]
        if cfg == "R1" and res["aksi"] != "jawab": continue
        ctx = res["konteks_teks"] if cfg == "R1" else gold_texts(QI.loc[x["id"], "konteks_acuan"])
        smp.append(SingleTurnSample(user_input=QI.loc[x["id"], "pertanyaan"], response=res["jawaban"],
                                    retrieved_contexts=ctx, reference=QI.loc[x["id"], "jawaban_referensi"]))
        ids.append(x["id"])
    return ids, smp

def run_metric(cfg, metric, nama):
    # satu metrik per panggilan; hasil disimpan ke CSV supaya tidak hilang bila sesi terputus
    fn = f"ragas_{cfg}_{nama}.csv"
    if os.path.exists(fn):
        print(cfg, nama, "sudah ada, dilewati"); return pd.read_csv(fn)
    ids, smp = make_samples(cfg)
    t0 = time.time()
    out = evaluate(EvaluationDataset(samples=smp), metrics=[metric], run_config=rc, raise_exceptions=False, show_progress=True)
    df = out.to_pandas()
    col = [c for c in df.columns if c not in ("user_input", "retrieved_contexts", "response", "reference")][0]
    d = pd.DataFrame({"id": ids, col: df[col].values})
    d.to_csv(fn, index=False)
    print(f"{cfg} {nama}: rata-rata={d[col].mean():.3f}  terisi={d[col].notna().sum()}/{len(d)}  ({time.time()-t0:.0f} dtk)")
    return d

# strictness=1: Gemini 3.x menolak 'multiple candidates' (n>1) yang dipakai default Ragas (strictness=3)
M = {
  "faithfulness":       lambda: Faithfulness(llm=judge),
  "answer_relevancy":   lambda: ResponseRelevancy(llm=judge, embeddings=lc_emb, strictness=1),
  "context_precision":  lambda: LLMContextPrecisionWithReference(llm=judge),
  "context_recall":     lambda: LLMContextRecall(llm=judge),
}
PLAN = [("B0", "faithfulness"), ("R1", "faithfulness"),
        ("B0", "answer_relevancy"), ("R1", "answer_relevancy"),
        ("R1", "context_precision"), ("R1", "context_recall")]
RES = {}
for cfg, nama in PLAN:
    try:
        RES[(cfg, nama)] = run_metric(cfg, M[nama](), nama)
    except Exception as e:
        print("GAGAL", cfg, nama, repr(e)[:200])

def gabung(cfg):
    ds = [RES[k].set_index("id") for k in RES if k[0] == cfg]
    return pd.concat(ds, axis=1).reset_index() if ds else None
RAGAS_B0, RAGAS_R1 = gabung("B0"), gabung("R1")
for cfg, df in (("B0", RAGAS_B0), ("R1", RAGAS_R1)):
    if df is not None:
        df.to_csv(f"ragas_{cfg}.csv", index=False); print(cfg, df.drop(columns="id").mean().round(3).to_dict())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (12 outputs left out)

Evaluating:   0%|          | 0/19 [00:00<?, ?it/s]

R1 context_precision: rata-rata=nan  terisi=0/19  (1256 dtk)


Evaluating:   0%|          | 0/19 [00:00<?, ?it/s]

R1 context_recall: rata-rata=nan  terisi=0/19  (1255 dtk)
B0 {'faithfulness': 0.167, 'answer_relevancy': nan}
R1 {'faithfulness': nan, 'answer_relevancy': nan, 'llm_context_precision_with_reference': nan, 'context_recall': nan}


In [ ]:
import os, time, getpass, warnings
warnings.filterwarnings("ignore")
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    os.environ.setdefault("GOOGLE_API_KEY", getpass.getpass("GOOGLE_API_KEY: "))
from google import genai
c = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
for m in ["gemini-3.5-flash-lite", "gemini-3.8-flash"]:
    t = time.time()
    try:
        r = c.models.generate_content(model=m, contents="Balas dengan satu kata: halo")
        print(m, "OK", round(time.time()-t, 1), "dtk ->", r.text.strip()[:40])
    except Exception as e:
        print(m, "GAGAL", round(time.time()-t, 1), "dtk ->", str(e)[:400])

gemini-3.5-flash-lite GAGAL 0.3 dtk -> 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, lim
gemini-3.8-flash GAGAL 13.4 dtk -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Ringkasan H1 sampai H5

In [ ]:
def p95(x): return float(np.percentile(x, 95))
R1 = RAW[RAW.konfigurasi == "R1"]; B0 = RAW[RAW.konfigurasi == "B0"]
ins = R1[R1.tipe == "dalam_cakupan"]; sen = R1[R1.tipe == "sensitif"]; oos = R1[R1.tipe == "di_luar_cakupan"]
get = lambda df, col: (float(df[col].mean()) if df is not None and col in df.columns and df[col].notna().any() else np.nan)

S = {}
S["H1_faithfulness_B0"] = get(RAGAS_B0, "faithfulness")
S["H1_faithfulness_R1"] = get(RAGAS_R1, "faithfulness")
S["H2_faithfulness_R1>=0.85"] = S["H1_faithfulness_R1"]
S["H1_answer_relevancy_B0"] = get(RAGAS_B0, "answer_relevancy")
S["H1_answer_relevancy_R1"] = get(RAGAS_R1, "answer_relevancy")
S["R1_context_precision"] = get(RAGAS_R1, "llm_context_precision_with_reference")
S["R1_context_recall"] = get(RAGAS_R1, "context_recall")
S["H3_latensi_p95_R1_semua_s"] = p95(R1.latensi_s)
S["H3_latensi_p95_R1_dalam_cakupan_s"] = p95(ins.latensi_s)
S["H3_latensi_rata2_R1_s"] = float(R1.latensi_s.mean())
S["H3_latensi_p95_B0_s"] = p95(B0.latensi_s)
S["H4_eskalasi_sensitif_tepat_%"] = 100 * float((sen.aksi == "eskalasi").mean())
S["H4_penolakan_di_luar_cakupan_tepat_%"] = 100 * float((oos.aksi == "tolak").mean())
S["H4_dalam_cakupan_dijawab_tanpa_agen_%"] = 100 * float((ins.aksi == "jawab").mean())
S["H5_token_per_pertanyaan_R1_dijawab"] = float(ins[ins.aksi == "jawab"].token_total.mean())
S["H5_token_per_pertanyaan_R1_semua"] = float(R1.token_total.mean())
S["H5_token_per_pertanyaan_B0"] = float(B0.token_total.mean())
SUM = pd.DataFrame({"metrik": list(S.keys()), "nilai": list(S.values())})
SUM.to_csv("ringkasan_poc.csv", index=False, encoding="utf-8-sig")
print(SUM.to_string(index=False))

# Distribusi skor retrieval (untuk pembahasan ambang)
print("\nSkor top-1 per tipe:\n", R1.groupby("tipe").skor_top1.describe()[["count", "min", "mean", "max"]])

NameError: name 'RAW' is not defined

In [ ]:
# Lembar penilaian manusia (isi kolom nilai_B0 dan nilai_R1: benar / sebagian / salah)
J = Q[["id", "tipe", "pertanyaan", "jawaban_referensi"]].copy()
rr = RAW.set_index(["id", "konfigurasi"])
J["jawaban_B0"] = [rr.loc[(i, "B0"), "jawaban"] for i in J.id]
J["jawaban_R1"] = [rr.loc[(i, "R1"), "jawaban"] for i in J.id]
J["aksi_R1"] = [rr.loc[(i, "R1"), "aksi"] for i in J.id]
J["nilai_B0"] = ""; J["nilai_R1"] = ""
J.to_csv("lembar_penilaian.csv", index=False, encoding="utf-8-sig")
print("lembar_penilaian.csv dibuat. Hanya pertanyaan 'dalam_cakupan' yang perlu dinilai.")

lembar_penilaian.csv dibuat. Hanya pertanyaan 'dalam_cakupan' yang perlu dinilai.


In [ ]:
# Catat ke MLflow (opsional)
try:
    import mlflow
    mlflow.set_experiment("poc-rag-bank")
    with mlflow.start_run(run_name="B0-vs-R1"):
        mlflow.log_params({"model": MODEL, "temperature": TEMPERATURE, "K": K, "chunk_tokens": CHUNK_TOKENS,
                           "overlap": OVERLAP, "score_threshold": SCORE_THRESHOLD, "embedding": "bge-m3",
                           "thinking": "default", "use_temperature": USE_TEMP})
        for k, v in S.items():
            if v == v: mlflow.log_metric(k.replace(">=", "ge").replace("%", "pct"), v)
        for f in ("hasil_mentah.csv", "ringkasan_poc.csv"): mlflow.log_artifact(f)
    print("MLflow tercatat")
except Exception as e:
    print("MLflow dilewati:", repr(e)[:150])

# Kemas dan unduh
with zipfile.ZipFile("hasil_poc.zip", "w") as z:
    for f in ("hasil_mentah.csv", "ringkasan_poc.csv", "lembar_penilaian.csv", "ragas_B0.csv", "ragas_R1.csv"):
        if os.path.exists(f): z.write(f)
try:
    from google.colab import files
    files.download("hasil_poc.zip")
except Exception:
    print("Unduh hasil_poc.zip dari panel File.")

2026/11/08 13:10:03 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/08 13:10:03 INFO mlflow.store.db.utils: Updating database tables
2026/11/08 13:10:05 INFO mlflow.tracking.fluent: Experiment with name 'poc-rag-bank' does not exist. Creating a new experiment.


MLflow tercatat


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>